# Week 7 · Tuesday: Autoencoders, RBM and the Keras Sequential API

Yesterday's network learned from labelled digits: every image came with the digit it shows. This morning's models learn from rows with no label at all, a ratings table and the digits themselves. This afternoon a network reads text. Today:

1. **Collaborative filtering with an RBM:** recommend a film from what similar users liked
2. **Build: an autoencoder that flags unusual digits:** the first full build of the day
3. **From TFLearn to Keras:** an old library read as history, and its Keras equivalents
4. **Composing models:** a whole model used as one layer of another
5. **Sequential composition:** three ways to build a straight stack, and where it stops
6. **Build: a text classifier with TF-IDF and a Dense network:** the second full build

Sections 2 and 6 are end-to-end builds. Their subsections follow the order your own project notebook follows: the problem, the dataset and the split, preprocessing, the model with a reason for each choice, training with both curves, evaluation, error analysis, one improvement attempt, the test set used once, and limitations.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

Run the next cell first. It holds the imports and plotting helpers we reuse all day, and it fixes the random seeds so your numbers match the ones on the screen.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow together
tf.get_logger().setLevel("ERROR")              # hide TensorFlow's retracing warnings; errors still show


def plot_history(history, metric="loss", title=""):
    """Draw the training and validation curves of one metric against the epoch, on one chart."""
    epochs = range(1, len(history.history[metric]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history[metric], marker="o", markersize=4, linewidth=2, label=f"training {metric}")
    plt.plot(epochs, history.history["val_" + metric], marker="o", markersize=4, linewidth=2,
             label=f"validation {metric}")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def show_image_grid(images, titles, columns=5):
    """Draw images (784-long rows or 28 by 28 by 1 arrays) as 28 by 28 pictures, in rows of `columns`."""
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(2.2 * columns, 2.4 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image, title in zip(axes.ravel(), images, titles):
        ax.imshow(np.reshape(image, (28, 28)), cmap="gray")
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


print(f"TensorFlow {tf.__version__}, Keras {tf.keras.__version__}, NumPy {np.__version__}, "
      f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}")
print("Setup complete")

## 1. Collaborative filtering with an RBM

A streaming service has a table of who liked what, and it wants to suggest a film you have not seen. **Collaborative filtering** recommends to a user what similar users liked, using only that table and no facts about the films themselves. The idea in one line: people who agreed before agree again.

### 1.1 A ratings table with blanks

The table below is **invented** for this notebook, so we know the right answer in advance. It has 8 users and 6 films. A 1 means the user liked the film. A 0 means the user did not like it **or has not seen it**: the table uses the same number for both.

- Four science-fiction fans, Ana, Ben, Cy and Dev, like Star Run, Moon Base and Robot Dawn.
- Four romance fans, Eve, Fay, Gus and Hal, like Love Letters, June Wedding and Paris Days.
- Each group has one **blank**: Dev has not seen Robot Dawn, and Hal has not seen Paris Days.

`np.array` holds the numbers the model reads. `pd.DataFrame` only adds the names so the table prints readably.

In [ ]:
FILMS = ["Star Run", "Moon Base", "Robot Dawn", "Love Letters", "June Wedding", "Paris Days"]
USERS = ["Ana", "Ben", "Cy", "Dev", "Eve", "Fay", "Gus", "Hal"]
ratings = np.array([
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],   # Dev has not seen Robot Dawn
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 0],   # Hal has not seen Paris Days
])
print(pd.DataFrame(ratings, index=USERS, columns=FILMS))

- Read it by rows. Dev agrees with Ana, Ben and Cy on Star Run and Moon Base, and all three of them liked Robot Dawn. Hal agrees with Eve, Fay and Gus on two films, and all three liked Paris Days.
- So a person would recommend Robot Dawn to Dev and Paris Days to Hal. That is the check the model has to pass in 1.3.
- The weakness of a 0 shows already: nothing in the table says whether Dev skipped Robot Dawn or watched it and disliked it.

**In an interview:** collaborative filtering recommends to a user what similar users liked. It needs only the table of who rated or liked what, and nothing about the items.

### 1.2 A restricted Boltzmann machine

A **restricted Boltzmann machine (RBM)** is a small two-layer network that learns a few hidden features from rows like ours, and can rebuild each row from them.

- The **visible units** hold the data: one per film, so a user's row of six ratings sits in six visible units.
- The **hidden units** are a few learned features. In a recommender each one plays the part of a **taste**. Every hidden unit is linked to every film, with one weight per link: 2 hidden units times 6 films is 12 weights.
- **Restricted** means no unit is linked to another unit in its own layer. That makes it cheap: all the hidden units can be worked out at once from a user's row, and all the films at once from the hidden units.

Keras has no RBM layer, so we use scikit-learn's `BernoulliRBM`. A Bernoulli variable takes only 0 or 1, and so do this model's units. Its arguments:

- `n_components`: the number of hidden units, our tastes.
- `learning_rate`: the size of each update to the weights.
- `n_iter`: the number of passes over the table.
- `random_state`: a fixed seed, so a rerun learns the same weights.

It trains by a method called **persistent contrastive divergence**. Recognise the name; we do not open it up. Notice that `fit` takes only the ratings. There is no label column.

In [ ]:
from sklearn.neural_network import BernoulliRBM

rbm = BernoulliRBM(n_components=2, learning_rate=0.1, n_iter=1000,   # Try changing it: n_components=1
                   random_state=RANDOM_STATE)
rbm.fit(ratings)                                 # no labels: the ratings are all it sees
print(f"components_ shape: {rbm.components_.shape}  (hidden units, films)")

weights = pd.DataFrame(rbm.components_, columns=FILMS,
                       index=[f"taste {i + 1}" for i in range(rbm.n_components)])
weights.loc["film bias"] = rbm.intercept_visible_       # one bias per film
print(weights.round(1))

- `components_` holds the weights, one row per hidden unit and one column per film: shape `(2, 6)`.
- Read a row by its signs. A positive weight means that when the hidden unit is on, it pulls the film's score up; a negative weight pushes it down.
- On this run both tastes have positive weights on the three romance films (about 2 to 4) and negative weights on the three science-fiction films. The model learned the same split twice, "romance, not science fiction". A science-fiction fan is then a user with both units off, and the **film bias** row (one number per film) carries their films: about 0.7 for Star Run and Moon Base, 0.0 for Robot Dawn, which one fan has not seen, and about -3.6 for each romance film.
- Nobody told the model "science fiction" or "romance". The rows are whatever patterns rebuild the table best, and we name them afterwards.
- Try changing it: `n_components=1`. `components_` becomes `(1, 6)`, one taste for the whole table. Run 1.3 with it: the two recommendations stay the same, because one hidden unit that is on for one group and off for the other is enough to tell two groups apart. That is also why our two units ended up alike. Put it back to 2 and rerun this cell.

**In an interview:** an RBM has a visible layer (here the films) and a hidden layer of learned features (here tastes), linked across the layers and never inside one. It turns a user's row into hidden tastes and can rebuild the row from them.

### 1.3 Recommending

To recommend, we rebuild every user's row from their hidden tastes and read the scores of the films they have not seen:

- `rbm.transform(ratings)` gives each user's **hidden-taste probabilities**, one number between 0 and 1 per taste: shape `(8, 2)`.
- `hidden @ rbm.components_` multiplies them by the weights (`@` is matrix multiplication): an 8-by-2 table times a 2-by-6 table gives an 8-by-6 table, one score per user and film. `+ rbm.intercept_visible_` adds each film's bias.
- `1 / (1 + np.exp(-(...)))` is the sigmoid from Friday. It squeezes each score into 0 to 1, the range of the ratings.

Then, for each user, we sort the films they rated 0 by their rebuilt score.

In [ ]:
hidden = rbm.transform(ratings)
rebuilt = 1 / (1 + np.exp(-(hidden @ rbm.components_ + rbm.intercept_visible_)))
print(f"hidden {hidden.shape}, rebuilt {rebuilt.shape}")

for user in ["Dev", "Hal"]:
    row = USERS.index(user)
    unseen = [film for film, rating in zip(FILMS, ratings[row]) if rating == 0]
    ranked = sorted(unseen, key=lambda film: rebuilt[row, FILMS.index(film)], reverse=True)
    scores = ", ".join(f"{film} {rebuilt[row, FILMS.index(film)]:.2f}" for film in ranked)
    print(f"{user}: recommend {ranked[0]}  (unseen films ranked: {scores})")

- Dev's top unseen film is Robot Dawn and Hal's is Paris Days: both blanks the table was built with come out on top, well clear of the other unseen films. Robot Dawn scores about 0.51 for Dev against 0.03 or less for the rest, and Paris Days about 0.67 for Hal against about 0.01.
- Dev's hidden tastes match Ana's, Ben's and Cy's, so Dev's rebuilt row looks like theirs, and theirs has Robot Dawn in it. The model never compared Dev with Ana directly. It turned each row into tastes and rebuilt the row from them.
- A brand-new user has an empty row, so there are no tastes to rebuild from. That is the **cold-start problem**, and services cover it with popular items or a few favourites asked for at sign-up.
- Where recommenders went: Keras has no RBM layer, and a team today would pick **matrix factorisation** (splitting the ratings table into a few numbers per user and a few per film, whose product rebuilds the table) or a deep recommender. The idea carries over: learn a few hidden features, rebuild the row, rank the blanks.

**In an interview:** to recommend with an RBM, turn a user's row into hidden tastes, rebuild the row from them, and rank the films the user has not rated by their rebuilt score. It cannot help a user with no ratings yet (cold start), and today teams use matrix factorisation or a deep recommender.

### 1.4 Each user's tastes

*Going deeper. The one idea to keep: the hidden tastes are a short description of each user, the way an autoencoder's code in section 2 is a short description of an image.*

`rbm.transform(ratings)` from 1.3, printed one user per row.

In [ ]:
tastes = pd.DataFrame(rbm.transform(ratings), index=USERS,
                      columns=[f"taste {i + 1}" for i in range(rbm.n_components)])
print(tastes.round(2))

- Each user is now two numbers. The science-fiction fans sit at 0.00 on both tastes and the romance fans at or near 1 (Hal at 0.98 and 0.96), so Dev looks like Ana, and Hal like Eve.
- They are probabilities that a hidden unit is on. They are not ratings of any film.

## 2. Build: an autoencoder that flags unusual digits

The first build of the day, in the project order. It uses Monday's digits, with one difference that changes everything: the labels are not used for training.

### 2.1 Problem statement and the idea

**Problem statement.** Flag the handwritten digits that do not look like the rest. Someone checking thousands of scanned forms wants the odd ones put in front of a person, and has no list of what "odd" looks like. The model gives each image one number, how badly it rebuilds it, and a flag when that number is too high.

**The idea.** An **autoencoder** is a network trained to rebuild its own input through a narrow layer in the middle:

- The **encoder** squeezes the 784 pixels into a short **code**, here 32 numbers.
- The **bottleneck** is that narrow middle layer. Its 32 numbers are all the second half gets.
- The **decoder** rebuilds all 784 pixels from the code.

The input is also the target, so no label is needed: the model learns what typical digits look like. An image it rebuilds badly is unlike what it trained on, the way a bank flags a transaction that does not fit a customer's usual pattern.

**In an interview:** an autoencoder learns to rebuild its input through a small bottleneck. The input is its own target, so it needs no labels, and an example it rebuilds badly is unusual compared with its training data.

### 2.2 The dataset and the split

**MNIST**, as on Monday: 70,000 handwritten digits of 28 by 28 pixels, loaded from code. The preprocessing is the week's usual two steps, done in the same cell: `reshape(-1, 784)` flattens each image to one row of 784 numbers (`-1` tells numpy to work out the number of rows), and dividing by 255.0 puts every pixel between 0 and 1.

The split is the one used all week: the first 54,000 training images to train on, the last 6,000 to validate on. **The test set is held out now and used once, at the end; every choice before then is made on the validation data.** `x_test` appears in exactly one cell of this build, in 2.9.

The labels `y_tr` and `y_val` are loaded but never used in training. They come back in 2.7, only to name the digits in the titles.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0     # flatten, then scale to 0..1
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

x_tr, y_tr = x_train[:54000], y_train[:54000]
x_val, y_val = x_train[54000:], y_train[54000:]
print(f"training {x_tr.shape}, validation {x_val.shape}, test {x_test.shape} (put aside)")
print(f"pixel range: {x_tr.min()} to {x_tr.max()}")

show_image_grid(x_tr[:10], [f"label {label}" for label in y_tr[:10]])

- 54,000 rows to train on and 6,000 to validate on, each 784 pixels wide. The 10,000 test rows wait for 2.9.
- Every pixel now lies between 0.0 and 1.0. That range decides the output layer in 2.3.
- The titles use the labels only so we can name what we see. The model will not get them.

**In an interview:** I put the test set aside first, used a fixed validation set for every choice, flattened each image to 784 numbers and scaled the pixels to 0 to 1.

### 2.3 The model and a reason for each choice

`autoencoder` is a `Sequential` of three parts: the `Input` of 784 pixels, an encoder `Dense(32, activation="relu")` that makes the code, and a decoder `Dense(784, activation="sigmoid")` that rebuilds the pixels. It is compiled with Adam and **MSE**, Monday's mean squared error: here, the mean squared difference between each pixel and its rebuild, called the **reconstruction loss**.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)      # the same starting weights on every rerun

autoencoder = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(32, activation="relu"),       # encoder: 784 pixels to a 32-number code
    tf.keras.layers.Dense(784, activation="sigmoid"),   # decoder: the code back to 784 pixels
])
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

The count, from Monday's rule (a Dense layer holds inputs times units, plus one bias per unit): the encoder has 784 × 32 + 32 = 25,120 and the decoder 32 × 784 + 784 = 25,872, so **50,992** in all.

| Choice | What we use | Why |
|---|---|---|
| Bottleneck | `Dense(32)` | far smaller than 784, so the network cannot copy the pixels through; wide enough to keep a digit's shape |
| Hidden activation | ReLU | lets the code bend with the data, as in Monday's hidden layer |
| Output layer | `Dense(784, activation="sigmoid")` | one unit per pixel; a sigmoid only gives values between 0 and 1, the range of our pixels |
| Loss | `"mse"` | the mean squared difference between the image and its rebuild; no label needed |
| Optimizer | `"adam"` | the week's optimizer, as on Monday |

Why the bottleneck must be smaller than the input: a middle layer as wide as the input lets the network pass the pixels straight through and learn nothing useful. One that is very narrow blurs the rebuilds.

Try changing it: `Dense(16, ...)` in the cell above, and read `summary()` only (no fit). The total falls to 25,888. Put it back to 32 and rerun the cell before 2.4.

**In an interview:** a 784-32-784 network with a ReLU bottleneck and a sigmoid output, because the pixels lie between 0 and 1, trained with mean squared error and Adam; the bottleneck is much smaller than the input so the model has to learn the shape of a digit.

### 2.4 Training with both curves

The fit line is the one to read: `autoencoder.fit(x_tr, x_tr, ...)`. The first two arguments are the inputs and the targets, and both are `x_tr`. The validation data is `(x_val, x_val)` for the same reason. We train 10 epochs with `batch_size=256` (256 images per weight update), and Keras prints one line per epoch with the training `loss` and the `val_loss`.

In [ ]:
history = autoencoder.fit(x_tr, x_tr, epochs=10, batch_size=256,   # the input is also the target
                          validation_data=(x_val, x_val), verbose=2)
plot_history(history, "loss", "Autoencoder reconstruction loss (MSE)")
print(f"epoch 1:  loss {history.history['loss'][0]:.4f}, val_loss {history.history['val_loss'][0]:.4f}")
print(f"epoch 10: loss {history.history['loss'][-1]:.4f}, val_loss {history.history['val_loss'][-1]:.4f}")

- Both curves fall fast, then flatten: the training loss drops from about 0.073 in epoch 1 to about 0.012 in epoch 10.
- The validation curve stays close to the training curve, ending at about 0.012 too. The model rebuilds digits it has never seen about as well as the ones it trained on: no sign of overfitting.
- If you write `fit(x_tr, y_tr)` by habit, the targets become digit labels and Keras stops with a shape error. The target of an autoencoder is the input.

**In an interview:** I trained it with the images as both input and target, passed the validation images the same way, and drew the training and validation loss on one chart to check it was learning and not overfitting.

### 2.5 Originals beside reconstructions

Before any number, look. `autoencoder.predict(x_val[:5])` rebuilds five validation digits, and `show_image_grid` draws the five originals on top and their rebuilds below.

In [ ]:
rebuilt_digits = autoencoder.predict(x_val[:5], verbose=0)      # Try changing it: x_val[5:10]
show_image_grid(np.concatenate([x_val[:5], rebuilt_digits]),
                [f"original {label}" for label in y_val[:5]] + ["rebuilt"] * 5)

- Each rebuild is clearly the same digit, a little softer. All the decoder gets is 32 numbers per image, and fine stroke detail does not survive the squeeze.
- A soft rebuild is what a 32-number bottleneck gives. The curves in 2.4 are the evidence that the model learned.
- Try changing it: `x_val[5:10]`, five other digits, just as soft. Put it back.

**In an interview:** I checked the rebuilds by eye against the originals before trusting any number: same digits, softer strokes.

### 2.6 Reconstruction error and a threshold

The **reconstruction error** of one image is the mean squared difference between its pixels and their rebuild: one number per image, and the score we flag on.

```
val_errors = np.mean((x_val - autoencoder.predict(x_val)) ** 2, axis=1)
```

`axis=1` averages along each row, across its 784 pixels, so there is one error per validation image.

In [ ]:
val_errors = np.mean((x_val - autoencoder.predict(x_val, verbose=0)) ** 2, axis=1)
print(f"val_errors shape {val_errors.shape}")
print(f"smallest {val_errors.min():.4f}, median {np.median(val_errors):.4f}, largest {val_errors.max():.4f}")

Now the **threshold**: the error above which we flag an image. We take the 99th percentile of the validation errors, the value below which 99 percent of them lie. About one validation image in a hundred lies above it, by construction.

We choose it on the validation errors because the model never trained on those images. Errors on training images are smaller than on new data, so a threshold set there would be too low and would flag too much. The threshold is a decision the team makes and writes down: the TensorFlow tutorial uses the mean plus one standard deviation of the training loss instead.

In [ ]:
threshold = np.percentile(val_errors, 99)       # Try changing it: 95
flagged = np.where(val_errors > threshold)[0]
print(f"threshold {threshold:.4f}: {len(flagged)} of {len(val_errors)} validation images above it")

plt.figure(figsize=(8, 5))
plt.hist(val_errors, bins=60)
plt.axvline(threshold, color="red", linewidth=2, label="threshold (99th percentile)")
plt.title("Reconstruction error per validation image")
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Images")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

- Most errors bunch at the low end, with a long thin tail to the right. The red line cuts off the far end of that tail.
- 60 of the 6,000 validation images are above the threshold: 1 percent, by construction. The count says nothing about how many digits are really odd. A percentile threshold flags a fixed share whatever the images look like.
- Try changing it: the 95th percentile. 300 images are flagged, 5 percent, and the red line moves left into the body of the histogram. A lower threshold catches more odd digits and raises more false alarms, the same trade as precision against recall in section 6. Put it back to 99 and rerun the cell before 2.7.

**In an interview:** I scored each image by its reconstruction error and set the threshold at the 99th percentile of the errors on validation data the model did not train on, because training errors are optimistic.

### 2.7 Error analysis: the digits it rebuilds worst

The images above the threshold are this build's errors: the model rebuilt them worst. Reading them says what the model finds unusual. First, which digits the 60 flagged images show, using the labels only to name them. Then the ten worst, with their rebuilds underneath.

`np.argsort(val_errors)` gives the positions of the errors from smallest to largest, `[::-1]` reverses that, and `[:10]` keeps the first ten: the ten worst.

In [ ]:
print("Flagged validation images per digit:")
print(pd.Series(y_val[flagged]).value_counts().sort_index().to_dict())

worst = np.argsort(val_errors)[::-1][:10]       # Try changing it: np.argsort(val_errors)[:10]
worst_rebuilt = autoencoder.predict(x_val[worst], verbose=0)
show_image_grid(np.concatenate([x_val[worst], worst_rebuilt]),
                [f"{y_val[i]}, error {val_errors[i]:.3f}" for i in worst] + ["rebuilt"] * 10, columns=5)

- Of the 60 flagged images, the most common digits are 8 (15 images) and 2 (14), then 5 (9): digits with loops and curves that people write in many ways. Not one 1 is flagged.
- The ten worst share a look more than a digit. Some have very thick, filled-in strokes (the two 0s, a 2, the 9). Others have unusual shapes: a 6 with a long tail, 8s drawn as two loose loops, a 2 with an extra loop and a stray mark beside it. Their rebuilds underneath are blurred toward a more typical shape, and some no longer read as the right digit. On Colab your ten may differ by an image or two; read them the same way.
- The labels named the digits but played no part in training, so nothing leaked.
- Try changing it: `np.argsort(val_errors)[:10]`, the ten best rebuilds. All ten are 1s with errors under 0.002: a thin straight stroke is the easiest shape to rebuild. Put it back.
- The limit: this autoencoder trained on every kind of digit, so it has learned to rebuild many shapes. An oddity it has partly learned gets a modest error and goes unflagged.

**In an interview:** I looked at the examples above the threshold and the worst rebuilds, and read them for what they share before changing anything.

### 2.8 One improvement attempt

One change: a bottleneck of 64 numbers instead of 32, fitted the same way on the same data and judged on the same validation images. The **results table** keeps one row per trained model. `evaluate(x_val, x_val)` returns the loss, and since the loss is MSE that is the mean reconstruction error over the validation images. We also keep each model's own 99th-percentile threshold, because a threshold belongs to one model.

In [ ]:
results = []
results.append({"model": "bottleneck 32", "val_loss": autoencoder.evaluate(x_val, x_val, verbose=0),
                "val_threshold": threshold})

tf.keras.utils.set_random_seed(RANDOM_STATE)
autoencoder_64 = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(64, activation="relu"),       # the one change: 64 numbers instead of 32
    tf.keras.layers.Dense(784, activation="sigmoid"),
])
autoencoder_64.compile(optimizer="adam", loss="mse")
autoencoder_64.fit(x_tr, x_tr, epochs=10, batch_size=256, validation_data=(x_val, x_val), verbose=0)

val_errors_64 = np.mean((x_val - autoencoder_64.predict(x_val, verbose=0)) ** 2, axis=1)
results.append({"model": "bottleneck 64", "val_loss": autoencoder_64.evaluate(x_val, x_val, verbose=0),
                "val_threshold": np.percentile(val_errors_64, 99)})
print(pd.DataFrame(results).round(4))

- It helped on the number we measured: the mean validation error falls from about 0.0119 to about 0.0056, less than half. 64 numbers hold more of each digit. The 64 model's own threshold falls too, from about 0.031 to about 0.016, because all its rebuilds are better.
- A lower error is not automatically a better detector. A model that rebuilds everything well rebuilds the odd digits well too, so the gap between a typical error and an unusual one can shrink. For flagging, what matters is how far the odd images stand out.
- One change at a time: change the bottleneck and the epochs together, and nobody can say which one moved the number.

We keep the row with the lower validation loss, chosen in one line before the test cell:

In [ ]:
kept_ae = autoencoder_64 if results[1]["val_loss"] < results[0]["val_loss"] else autoencoder
kept_threshold = results[1]["val_threshold"] if kept_ae is autoencoder_64 else results[0]["val_threshold"]
print(f"kept: {kept_ae.layers[0].units}-number bottleneck, threshold {kept_threshold:.4f}")

**In an interview:** I tried one change, a 64-number bottleneck, on the same validation images, kept the model with the lower validation error, and noted that a lower error is not always a better detector.

### 2.9 The test set, once, and limitations

The test cell runs once, on the kept model with its own threshold. It prints the share of test images above the threshold, then scores five **inverted** digits, `1.0 - x_test[:5]`: white digits on black become black digits on white, images unlike anything in training.

In [ ]:
test_errors = np.mean((x_test - kept_ae.predict(x_test, verbose=0)) ** 2, axis=1)
print(f"test images above the threshold: {(test_errors > kept_threshold).mean():.4f} "
      f"({(test_errors > kept_threshold).sum()} of {len(test_errors)})")

inverted = 1.0 - x_test[:5]                     # black on white: unlike every training image
inverted_errors = np.mean((inverted - kept_ae.predict(inverted, verbose=0)) ** 2, axis=1)
print(f"inverted digits' errors: {inverted_errors.round(3)}")
print(f"flagged: {inverted_errors > kept_threshold}")
show_image_grid(inverted, [f"error {e:.3f}" for e in inverted_errors])

- 68 of the 10,000 test images, about 0.7 percent, are above the kept model's threshold. The test digits are the same kind as the training digits, so the share lands close to the 1 percent of the validation images.
- All five inverted digits are flagged, with errors of about 0.8 to 0.9, between fifty and sixty times the threshold of about 0.016.
- Teams use this pattern for unusual transactions, sensor faults and compression. A variant, the **denoising autoencoder**, is trained to rebuild a clean image from a noisy copy; Friday's page builds one.

**Limitations.** The model flags what is unusual compared with its training digits, not what is wrong, and an oddity it has partly learned to rebuild slips under the threshold. A percentile threshold flags a fixed share of images whether or not anything odd is present.

**In an interview** ("Walk me through a model you built"): I built an autoencoder to flag unusual handwritten digits. I put the MNIST test set aside, flattened and scaled the images, and built a 784-32-784 network with a sigmoid output and MSE loss, trained on the images as their own targets with both loss curves on one chart. I scored each image by its reconstruction error, set the threshold at the 99th percentile of the validation errors, and read the worst rebuilds. I tried a 64-number bottleneck on the same validation data, used the test set once, and named two limitations.

## 3. From TFLearn to Keras

### 3.1 TFLearn

**TFLearn** was a high-level library that made TensorFlow 1 easier to use. It turns up in old tutorials, blog posts and course material, so it is worth reading. It is not worth running:

- Its last release on PyPI is 0.5.0, from 11 November 2020, and the last commit on its `master` branch is from 30 November 2020 (checked on 2026-11-18). It is unmaintained.
- It was built on TensorFlow 1's **graph mode**, where code first describes a computation and a session runs it later: the part TensorFlow 2 replaced.
- It is expected to fail to import on Colab's TensorFlow 2.21. Its file `tflearn/layers/recurrent.py` imports a helper called `is_sequence` from `tensorflow.python.util.nest`, a name TensorFlow 2.21 no longer defines (the cell below checks it), and its `__init__.py` switches TensorFlow 2 behaviour off with `tf.disable_v2_behavior()`.

We never install it. The cell below only asks TensorFlow whether the helper TFLearn needs is still there:

In [ ]:
from tensorflow.python.util import nest          # the module TFLearn imports from
print(f"TensorFlow {tf.__version__}: does nest.is_sequence exist? {hasattr(nest, 'is_sequence')}")

It is missing. TFLearn's source imports `is_sequence` from `tensorflow.python.util.nest`, so `import tflearn` is expected to fail on that line here. That failure would be in the library, so a typo fix or a `pip install` would not help.

Here is a TFLearn digit network, the kind you will meet in an old tutorial. Each line takes `net`, adds one layer and hands the result to the next line. `shape=[None, 784]` means any number of rows of 784 pixels. `regression` attaches the training settings, `DNN` wraps the whole graph in a model, and `fit` trains it.

**TFLearn (history, do not run)**

```python
import tflearn

net = tflearn.input_data(shape=[None, 784])
net = tflearn.fully_connected(net, 128, activation='relu')
net = tflearn.fully_connected(net, 10, activation='softmax')
net = tflearn.regression(net)
model = tflearn.DNN(net)
model.fit(X, Y)
```

Despite its name, `tflearn.regression` does not make a regression model: it plays the part of Keras's `compile`, and the network above is a digit classifier.

**In an interview:** an old TFLearn tutorial is expected to fail on current TensorFlow: TFLearn was written for TensorFlow 1's graph mode, and its source imports `is_sequence`, a name TensorFlow 2.21 no longer defines. I read it for the idea and write the Keras version.

### 3.2 The same network in Keras

The same network as Monday's digit classifier, 784-128-10, built and compiled. We do not fit it: the point is the translation, and Monday fitted this shape.

In [ ]:
digit_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),                         # tflearn.input_data
    tf.keras.layers.Dense(128, activation="relu"),        # tflearn.fully_connected   Try changing it: 64
    tf.keras.layers.Dense(10, activation="softmax"),      # tflearn.fully_connected
])
digit_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",   # tflearn.regression
                    metrics=["accuracy"])
digit_model.summary()

- **101,770** parameters, Monday's figure: 784 × 128 + 128 = 100,480 for the hidden layer and 128 × 10 + 10 = 1,290 for the output.
- Try changing it: `Dense(64, ...)`. The total becomes 50,890. Put it back.

Each TFLearn call has a Keras equivalent:

| TFLearn | Keras (TensorFlow 2) |
|---|---|
| `tflearn.input_data` | `tf.keras.Input` |
| `tflearn.fully_connected` | `tf.keras.layers.Dense` |
| `tflearn.regression` | `model.compile` |
| `tflearn.DNN(...)` and its `fit` | `model.fit` |
| TFLearn's layer stacking | `tf.keras.Sequential` |
| `tflearn.merge` | `tf.keras.layers.Concatenate` in the Functional API (Wednesday) |
| `tflearn.batch_normalization` | `tf.keras.layers.BatchNormalization` (Wednesday) |
| `DNN.save` and `DNN.load` | `model.save("....keras")` and `tf.keras.models.load_model` |
| `DNN(tensorboard_verbose=..., tensorboard_dir=...)` | `tf.keras.callbacks.TensorBoard(log_dir=...)` |

Keras is TensorFlow's own high-level API, reached as `tf.keras`, with nothing to install.

**In an interview** ("What is TFLearn, and what would you use today?"): a high-level library that made TensorFlow 1 easier to use and stopped at release 0.5.0 in 2020. Today Keras, TensorFlow's own high-level API, does the same job.

## 4. Composing models

### 4.1 An encoder and a decoder as two models

In Keras **a model is a layer**: a whole model can sit inside another model as one of its layers. We rebuild section 2's autoencoder from two parts, each a `Sequential` with a name:

- `encoder`: 784 pixels in, a 32-number code out.
- `decoder`: a 32-number code in, 784 pixels out. Its `Input(shape=(32,))` matches the encoder's 32 outputs, which is what lets the two join.
- `composed = Sequential([encoder, decoder])`: a stack whose two "layers" are whole models.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
encoder = tf.keras.Sequential([tf.keras.Input(shape=(784,)),
                               tf.keras.layers.Dense(32, activation="relu")], name="encoder")
decoder = tf.keras.Sequential([tf.keras.Input(shape=(32,)),
                               tf.keras.layers.Dense(784, activation="sigmoid")], name="decoder")
composed = tf.keras.Sequential([encoder, decoder], name="composed")
composed.compile(optimizer="adam", loss="mse")
composed.summary()

- `summary()` shows the two models as two rows, `encoder` with 25,120 parameters and `decoder` with 25,872: 50,992 in all, the same network as section 2.

Training `composed` trains its parts, because the parts inside it are the very same objects. Five epochs, the same data as section 2:

In [ ]:
history_composed = composed.fit(x_tr, x_tr, epochs=5, batch_size=256,
                                validation_data=(x_val, x_val), verbose=0)
print(f"val_loss after 5 epochs: {history_composed.history['val_loss'][-1]:.4f}")
print(f"composed.get_layer('encoder') is encoder: {composed.get_layer('encoder') is encoder}")

- The validation loss after 5 epochs is about 0.018, the same as section 2's `Epoch 5/10` line: the same network from the same seed, trained on the same data.
- `get_layer("encoder") is encoder` prints `True`: the encoder inside `composed` is the same object as our `encoder` variable, so it is trained now.
- If you change the encoder's width, change the decoder's `Input(shape=(32,))` to match, or the two parts no longer join.

**In an interview:** yes, a Keras model can be a layer of another model. I built an encoder and a decoder as two named models and stacked them, and training the whole trains both parts because they share the weights.

### 4.2 Using one part alone

Because the parts are models, each one runs alone. `encoder.predict` turns digits into codes, and `decoder.predict` turns codes back into images.

In [ ]:
codes = encoder.predict(x_val[:5], verbose=0)           # Try changing it: x_val[10:15]
images = decoder.predict(codes, verbose=0)
print(f"codes {codes.shape}, images {images.shape}")
print(f"first digit's code: {codes[0].round(1)}")
show_image_grid(np.concatenate([x_val[:5], images]), ["original"] * 5 + ["from its code"] * 5)

- Five digits become five codes of 32 numbers, shape `(5, 32)`: about 24 times fewer numbers than the 784 pixels. The decoder turns them back into five images, shape `(5, 784)`.
- Some of the 32 numbers are exactly 0: the ReLU turns every negative value into 0.
- The encoder alone is a compressor and a feature maker: its codes can be stored, compared or fed to another model. Calling `decoder.predict` on images instead of codes fails, because the decoder expects 32 numbers per row.
- Try changing it: `x_val[10:15]`. The same shapes, five other digits. Put it back.

**In an interview:** to use only the encoder of a trained autoencoder, I build it as its own model, nest it in the autoencoder, and after training call `encoder.predict` to get the codes.

### 4.3 A classifier on the codes

*Going deeper. The one idea to keep: a trained part can be reused under a new output layer, and freezing it keeps its weights fixed. Reusing a trained part for a new task is called **transfer learning**.*

`encoder.trainable = False` freezes the encoder. The change takes effect when the model is compiled afterwards, so we set it first. Then a new softmax layer on top learns to name the digit from its 32-number code, using the labels this time, for 3 epochs.

In [ ]:
encoder.trainable = False                      # freeze it before compile
tf.keras.utils.set_random_seed(RANDOM_STATE)
classifier = tf.keras.Sequential([encoder, tf.keras.layers.Dense(10, activation="softmax")])
classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

trainable = sum(int(np.prod(w.shape)) for w in classifier.trainable_weights)
frozen = sum(int(np.prod(w.shape)) for w in classifier.non_trainable_weights)
print(f"weights that learn: {trainable}, frozen: {frozen}")

history_codes = classifier.fit(x_tr, y_tr, epochs=3, batch_size=128,
                               validation_data=(x_val, y_val), verbose=0)
print(f"validation accuracy after 3 epochs: {history_codes.history['val_accuracy'][-1]:.3f}")

- Only the new layer learns: 32 × 10 + 10 = 330 weights. The encoder's 25,120 stay frozen.
- The validation accuracy is about 86%, from codes learned only for rebuilding digits. Monday's network, trained on the labels from the pixels up, reached about 97.6% on the same validation rows: the frozen codes keep what helps a rebuild, which is not always what separates the digits best.

## 5. Sequential composition

### 5.1 Three ways to build one Sequential

A **Sequential** model is a plain stack of layers in which each layer has one input and one output and feeds the next. Every Keras model today has been one. There are three ways to write the same stack:

- **A list** passed to `tf.keras.Sequential([...])`: the usual form, because the whole architecture reads top to bottom in one place.
- **`.add()`**: start from an empty `tf.keras.Sequential()` and add one layer at a time. You see it where layers are added in a loop or under a condition.
- **A list followed by `.add()`**: start from part of the stack and add the rest.

Each one can be read back with `model.layers`, the list of its layers in order.

In [ ]:
listed = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
], name="listed")

added = tf.keras.Sequential(name="added")
added.add(tf.keras.Input(shape=(784,)))
added.add(tf.keras.layers.Dense(128, activation="relu"))    # Try changing it: swap this line and the next
added.add(tf.keras.layers.Dense(10, activation="softmax"))

mixed = tf.keras.Sequential([tf.keras.Input(shape=(784,)),
                             tf.keras.layers.Dense(128, activation="relu")], name="mixed")
mixed.add(tf.keras.layers.Dense(10, activation="softmax"))

for model in [listed, added, mixed]:
    units = [layer.units for layer in model.layers]
    print(f"{model.name:7} layers {units}, parameters {model.count_params()}")
added.summary()

- All three print layers `[128, 10]` and 101,770 parameters: the same architecture, and the same `summary()`.
- `model.layers` lists the Dense layers only. The `Input` is a declaration of the shape, not a layer that holds weights.

Order matters, and Keras will not always tell you. A Keras 3 `Dense` layer declares only its number of units, and builds its weights from whatever width arrives. So `Dense(10)` before `Dense(128)` raises no error: it builds a different model. The only reorder that fails is an `Input` placed after a layer, and it fails when data first goes through.

In [ ]:
swapped = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(10, activation="softmax"),
    tf.keras.layers.Dense(128, activation="relu"),
])
print(f"swapped: parameters {swapped.count_params()}, output shape {swapped.predict(x_val[:1], verbose=0).shape}")

late_input = tf.keras.Sequential([tf.keras.layers.Dense(128, activation="relu"),
                                  tf.keras.Input(shape=(784,))])
try:
    late_input(x_val[:1])                      # the first data through the model
except ValueError as err:
    print(f"late Input: ValueError: {str(err).split('. ')[0]}.")

- The swapped model builds without a word: 784 × 10 + 10 = 7,850 plus 10 × 128 + 128 = 1,408, so **9,258** parameters, and an output 128 wide where a digit classifier needs 10. Only `summary()` or the output shape shows the mistake.
- The model with its `Input` after a layer raises a `ValueError` as soon as data goes through it.
- Try changing it: swap the two `.add(Dense(...))` lines in the cell before, and read the new parameter count. Put them back and rerun it.

**In an interview:** a Sequential model is a straight stack of layers, each with one input and one output. You can build it from a list, with `.add()`, or both, and read it back with `model.layers`; I start every stack with `tf.keras.Input` and check `summary()`, because swapped Dense layers build without an error.

### 5.2 Where Sequential stops

Sequential covers a stack with one input and one output, which is a large share of models, including every one in this notebook. Section 4's composed model was still a straight line: encoder, then decoder. Sequential stops at:

- **Several inputs**, such as a photo and a table of numbers about it.
- **Several outputs**, such as a class and a confidence score.
- **A shared layer**, one layer used by two paths.
- **A branch, a merge or a skip** around a layer.

Each of those needs layers wired by hand instead of stacked. Wednesday does that with the **Functional API**.

**In an interview:** I cannot use a Sequential model when the model has several inputs or outputs, shares a layer, or branches, merges or skips; then I use the Functional API.

## 6. Build: a text classifier with TF-IDF and a Dense network

The second build of the day, in the same project order, on text this time.

### 6.1 Problem statement

**Problem statement.** Route a forum post to the cars board or the motorcycles board from its text. A moderator, or a support desk that sorts tickets by hand all day, would use it to send each new post to the right place. The model reads a post and gives the probability that it belongs on the motorcycles board; a threshold turns that into a board.

**In an interview:** the problem was routing a forum post to one of two boards, cars or motorcycles, from its text alone, for a moderator or a support desk sorting posts by hand.

### 6.2 The dataset and the split

**20 Newsgroups** is a collection of old forum posts in 20 topics. scikit-learn's `fetch_20newsgroups` downloads it the first time it runs (about 14 MB) and keeps a copy on the runtime. We take two topics, `rec.autos` and `rec.motorcycles`. Two arguments matter:

- `subset="train"` and `subset="test"`: the collection comes already split. The test posts are our test set, put aside here. **The test set is held out now and used once, at the end; every choice before then is made on the validation data.**
- `remove=("headers", "footers", "quotes")`: strips the email headers, signatures and quoted replies, so the model reads the post and not the sender's signature.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

TOPICS = ["rec.autos", "rec.motorcycles"]
clean = ("headers", "footers", "quotes")
train = fetch_20newsgroups(subset="train", categories=TOPICS, remove=clean, random_state=RANDOM_STATE)
test = fetch_20newsgroups(subset="test", categories=TOPICS, remove=clean, random_state=RANDOM_STATE)
labels_test = test.target                       # put aside until 6.9

print(f"topics {train.target_names}  (label 0, label 1)")
print(f"training posts {len(train.data)}, test posts {len(test.data)} (put aside)")
print(f"training posts per topic: {np.bincount(train.target)}")
print(f"training posts left empty by remove: {sum(1 for text in train.data if not text.strip())}")

`rec.motorcycles` is label 1, the **positive class**: precision and recall below are about it. Now the training posts are split into training and validation with Monday's `train_test_split`. `stratify` keeps the share of each topic the same in both parts.

In [ ]:
from sklearn.model_selection import train_test_split

texts_tr, texts_val, labels_tr, labels_val = train_test_split(
    train.data, train.target, test_size=0.2, stratify=train.target, random_state=RANDOM_STATE)
print(f"training {len(texts_tr)}, validation {len(texts_val)}")
print(f"validation posts per topic: {np.bincount(labels_val)}")

for topic in [0, 1]:                            # the first post of each topic with a few sentences in it
    i = next(i for i, text in enumerate(texts_tr)
             if labels_tr[i] == topic and len(text.strip()) > 150 and "[...]" not in text)
    print(f"\n[{TOPICS[topic]}] {' '.join(texts_tr[i].split())[:250]}")

- 1,192 training posts, 594 about cars and 598 about motorcycles, and 794 test posts put aside. The split leaves 953 posts to train on and 239 to validate on, 119 and 120 per topic.
- The posts are short and informal, and some are only a line or two. 47 training posts were nothing but headers, signatures and quotes, so `remove` left them empty. With those gone, the words of the post are all the model gets.

**In an interview:** I used the collection's own test posts as the test set, put aside first, split the training posts 80/20 into training and validation with the topics stratified, and removed headers, signatures and quotes so the model reads the post.

### 6.3 Preprocessing

A network reads numbers, so each post becomes a row of word scores with W6D3's **TF-IDF**: a word scores high when it is frequent in this post and low when it appears in almost every post ("the", "and", "it"), so the words that separate the topics carry the most weight.

- `TfidfVectorizer(max_features=5000)` keeps the 5,000 words most frequent across the training posts: the **vocabulary**, one column each.
- `fit_transform(texts_tr)` learns the vocabulary from the **training posts only**. `transform` scores the validation and test posts against that same vocabulary, so all three tables share the same 5,000 columns. Fitting on validation or test posts would let the posts that judge the model shape its features, a leak.
- `transform` returns a sparse table (only the non-zero scores are stored). `.toarray().astype("float32")` makes it a plain dense table of 32-bit floats, which is what the network reads.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X_tr = vectorizer.fit_transform(texts_tr).toarray().astype("float32")    # learn the vocabulary here only
X_val = vectorizer.transform(texts_val).toarray().astype("float32")
X_test = vectorizer.transform(test.data).toarray().astype("float32")    # scored now, used once in 6.9

print(f"vocabulary size {len(vectorizer.vocabulary_)}")
print(f"X_tr {X_tr.shape} {X_tr.dtype}, X_val {X_val.shape}, X_test {X_test.shape}")
print(f"X_tr takes {X_tr.nbytes / 1e6:.1f} MB")

- 5,000 columns in every table, because every table uses the training vocabulary. A word in a validation post that is not in that vocabulary is simply left out.
- The dense training table is about 19 MB (953 rows × 5,000 columns × 4 bytes), small for a CPU runtime.

**In an interview:** I turned each post into TF-IDF scores over a 5,000-word vocabulary fitted on the training posts only, then scored the validation and test posts with the same vectorizer.

### 6.4 The model and a reason for each choice

`text_model` is 5000-32-1: an input as wide as the vocabulary, one hidden ReLU layer of 32 units, and one sigmoid output.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
text_model = tf.keras.Sequential([
    tf.keras.Input(shape=(5000,)),                      # one input per vocabulary word
    tf.keras.layers.Dense(32, activation="relu"),       # Try changing it: 16 (summary only)
    tf.keras.layers.Dense(1, activation="sigmoid"),     # the probability of rec.motorcycles
])
text_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
text_model.summary()

**160,065** parameters: 5,000 × 32 + 32 = 160,032 for the hidden layer and 32 + 1 = 33 for the output.

| Choice | What we use | Why |
|---|---|---|
| Input | `Input(shape=(5000,))` | it must equal the number of TF-IDF columns |
| Hidden layer | `Dense(32, activation="relu")` | a small layer: about 950 posts cannot support a big one |
| Output layer | `Dense(1, activation="sigmoid")` | two classes need one probability, of label 1 |
| Loss | `"binary_crossentropy"` | Monday's cross-entropy for a yes-or-no label |
| Optimizer | `"adam"` | the week's optimizer |

160,065 weights is a lot for about 950 posts, so watch 6.5's chart for a validation loss that turns upwards.

**In an interview:** a Dense network with one hidden ReLU layer and one sigmoid output, trained with binary cross-entropy and Adam, with an input as wide as the TF-IDF vocabulary.

### 6.5 Training with both curves

10 epochs with `batch_size=32`: about 30 weight updates per epoch on about 950 posts. The validation data is the validation posts and their labels.

In [ ]:
history_text = text_model.fit(X_tr, labels_tr, epochs=10, batch_size=32,
                              validation_data=(X_val, labels_val), verbose=2)
plot_history(history_text, "loss", "Text model loss (binary cross-entropy)")
best_epoch = int(np.argmin(history_text.history["val_loss"])) + 1
print(f"lowest val_loss {min(history_text.history['val_loss']):.3f} at epoch {best_epoch}; "
      f"epoch 10: loss {history_text.history['loss'][-1]:.3f}, val_loss {history_text.history['val_loss'][-1]:.3f}")

- The training loss falls fast, from about 0.69 to about 0.15, and the training accuracy reaches about 98%.
- The validation loss falls more slowly, to about 0.38, with a validation accuracy of about 85%. It is still falling at epoch 10, so these 10 epochs have not overfitted by the validation curve.
- The gap between the two curves is the warning from 6.4: 160,065 weights for 953 posts fit the training posts much better than new ones.

**In an interview:** I trained for 10 epochs with the validation posts passed in, drew both loss curves on one chart and read where the validation loss stopped falling.

### 6.6 Evaluation on the validation data

`predict` returns one column per output unit, a probability of motorcycles. `[:, 0]` takes the single column, and `>= 0.5` turns each probability into a label: the **0.5 threshold** is the usual default for a binary model.

- **Precision**: of the posts called motorcycles, the share that were.
- **Recall**: of the motorcycle posts, the share the model found.
- **F1**: precision and recall combined into one number, high only when both are.

`precision_score`, `recall_score` and `f1_score` take the true labels first and the predictions second. In the **confusion matrix**, rows are the true labels and columns the predictions, so the right answers sit on the diagonal.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

val_prob = text_model.predict(X_val, verbose=0)[:, 0]     # the probability of motorcycles, one per post
val_pred = (val_prob >= 0.5).astype(int)                  # Try changing it: 0.3
print(f"accuracy  {accuracy_score(labels_val, val_pred):.3f}")
print(f"precision {precision_score(labels_val, val_pred):.3f}")
print(f"recall    {recall_score(labels_val, val_pred):.3f}")
print(f"F1        {f1_score(labels_val, val_pred):.3f}")

matrix = confusion_matrix(labels_val, val_pred)
tn, fp, fn, tp = matrix.ravel()
print(f"true negatives {tn}, false positives {fp}, false negatives {fn}, true positives {tp}")
ConfusionMatrixDisplay(matrix, display_labels=TOPICS).plot(cmap="Blues")
plt.show()

- Accuracy is about 0.85. For motorcycles, precision is about 0.87 and recall about 0.81, so F1 is about 0.84.
- The matrix shows which way the mistakes go: the model misses more motorcycle posts (23 false negatives) than it wrongly calls motorcycles (14 false positives).
- For two classes, `ravel()` returns true negatives, false positives, false negatives and true positives, in that order.
- Try changing it: `>= 0.3` instead of `>= 0.5`. More posts are called motorcycles, so recall rises to about 0.95 and precision falls to about 0.71: the false negatives drop from 23 to 6 and the false positives rise from 14 to 47. Put it back to 0.5 and rerun the cell before 6.7, which reads these predictions.

**In an interview:** on the validation posts I reported accuracy, precision, recall and F1 for the motorcycles class and read the confusion matrix to see which way the mistakes went.

### 6.7 Error analysis

The numbers say how many posts are wrong. The posts themselves say why. A **false positive** is a car post the model sent to motorcycles; a **false negative** is a motorcycle post it sent to cars. `&` keeps the positions where both conditions hold, and `np.where(...)[0]` turns them into indexes. We print the first 300 characters of three of each.

In [ ]:
false_pos = np.where((val_pred == 1) & (labels_val == 0))[0]
false_neg = np.where((val_pred == 0) & (labels_val == 1))[0]
print(f"{len(false_pos)} false positives, {len(false_neg)} false negatives")

for kind, rows in [("FALSE POSITIVE (a car post called motorcycles)", false_pos[:3]),   # Try changing it: [:5]
                   ("FALSE NEGATIVE (a motorcycle post called cars)", false_neg[:3])]:
    for i in rows:
        text = " ".join(texts_val[i].split()) or "(empty: nothing left after remove)"   # one line
        print(f"\n{kind}, p = {val_prob[i]:.2f}:\n{text[:300]}")

`p` is the model's probability of motorcycles for that post. On Colab your six may differ by a post or two, because a post near 0.5 can land on the other side; read the ones on your screen the same way.

- Two of the false positives ask about a Honda Prelude, a car. Honda also makes motorcycles, so the brand name pulls the post toward the wrong board. The third is a rant about tailgating a truck: road talk that fits either board.
- The false negatives include a reply about an April Fool's joke with no vehicle words at all, and a post about who is liable for a car's photo-radar ticket, written on the motorcycles board. The third is empty: `remove` left nothing, so the model sees a row of zeros and answers about 0.5.
- What they share: words both boards use (brands, roads, tickets, insurance), posts with no topic words, and posts with no text. A word-score model has least to go on there, and no change to the network fixes an empty post.

**In an interview:** I read the misrouted posts as text, three of each kind, and found that most used words both boards share, such as brand names and road talk, or had almost no text to go on.

### 6.8 One improvement attempt

One change: a bigger vocabulary, `max_features=20000`. A new vectorizer, fitted on the training posts only, and a model of the same shape whose input width follows the new number of columns. Copying `Input(shape=(5000,))` unchanged is the shape error people hit here. Both rows of the results table are measured on the same validation posts. `evaluate` returns the loss first and the accuracy second, hence the `[0]`.

In [ ]:
results = [{"model": "max_features=5000", "val_loss": text_model.evaluate(X_val, labels_val, verbose=0)[0],
            "val_f1": f1_score(labels_val, val_pred)}]

vectorizer_20k = TfidfVectorizer(max_features=20000)
X_tr_20k = vectorizer_20k.fit_transform(texts_tr).toarray().astype("float32")   # training posts only
X_val_20k = vectorizer_20k.transform(texts_val).toarray().astype("float32")
print(f"vocabulary size {len(vectorizer_20k.vocabulary_)}, X_tr_20k {X_tr_20k.shape}, {X_tr_20k.nbytes / 1e6:.1f} MB")

tf.keras.utils.set_random_seed(RANDOM_STATE)
model_20k = tf.keras.Sequential([
    tf.keras.Input(shape=(X_tr_20k.shape[1],)),        # the input width follows the vocabulary
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model_20k.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_20k.fit(X_tr_20k, labels_tr, epochs=10, batch_size=32, validation_data=(X_val_20k, labels_val), verbose=0)

pred_20k = (model_20k.predict(X_val_20k, verbose=0)[:, 0] >= 0.5).astype(int)
results.append({"model": "max_features=20000", "val_loss": model_20k.evaluate(X_val_20k, labels_val, verbose=0)[0],
                "val_f1": f1_score(labels_val, pred_20k)})
print(pd.DataFrame(results).round(4))

- The training posts hold only 11,416 different words, so `max_features=20000` kept all of them: the vocabulary grew to 11,416 columns, not 20,000, and the input width followed.
- It helped a little: validation F1 rose from about 0.840 to about 0.858, and the validation loss fell from about 0.378 to about 0.359. On 239 posts that is a few posts more right, so read a gap this small with care.
- A bigger vocabulary keeps rarer words, and adds columns and fit time: the training table grew from about 19 MB to about 44 MB.

We keep the row with the higher validation F1, chosen in one line before the test cell:

In [ ]:
use_20k = results[1]["val_f1"] > results[0]["val_f1"]
kept_text, kept_vectorizer = (model_20k, vectorizer_20k) if use_20k else (text_model, vectorizer)
print(f"kept: {'max_features=20000' if use_20k else 'max_features=5000'}")

**In an interview:** I tried one change, a 20,000-word vocabulary with the input width to match, measured it on the same validation posts, and kept the model with the higher validation F1.

### 6.9 The test set, once, and limitations

The test posts go through the kept model's own vectorizer, so the table has the columns that model expects. This cell runs once. Its numbers are the ones to report, because no choice was made by looking at them.

In [ ]:
if kept_vectorizer is vectorizer:
    X_test_kept = X_test                        # already scored in 6.3
else:
    X_test_kept = kept_vectorizer.transform(test.data).toarray().astype("float32")   # the kept model's columns
test_pred = (kept_text.predict(X_test_kept, verbose=0)[:, 0] >= 0.5).astype(int)
print(f"test posts {len(labels_test)}: accuracy {accuracy_score(labels_test, test_pred):.3f}, "
      f"precision {precision_score(labels_test, test_pred):.3f}, recall {recall_score(labels_test, test_pred):.3f}, "
      f"F1 {f1_score(labels_test, test_pred):.3f}")

- On the 794 test posts the kept model scores about 0.87 accuracy, 0.89 precision, 0.84 recall and 0.86 F1, close to its validation F1 of about 0.86. No choice was made on these posts, so this is the honest estimate of how it does on new ones.

**Limitations.** The model separates two topics only, and a post about anything else is still forced onto one of the two boards. TF-IDF ignores word order, so "the car hit the bike" and "the bike hit the car" get the same scores; Friday's page tries an `Embedding` layer on the same question.

**In an interview** ("Walk me through a model you built"): I built a classifier that routes forum posts to the cars or the motorcycles board. I put the test posts aside first, split the rest into training and validation, and turned each post into TF-IDF scores with a vectorizer fitted on the training posts only. A small Dense network with one sigmoid output learned from them, with both loss curves on one chart. I reported precision, recall and F1 with a confusion matrix, read the misrouted posts, tried a bigger vocabulary on the same validation posts, used the test set once and named two limitations.

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Collaborative filtering with an RBM | unsupervised, `BernoulliRBM` in scikit-learn | recommending from a ratings table alone: hidden tastes rebuild each row | cold start (no ratings, no tastes); a 0 means not liked or not seen; Keras has no RBM layer, teams now use matrix factorisation |
| Autoencoder as an anomaly detector | unsupervised, Keras `Sequential` 784-32-784 | flagging images it rebuilds badly, with no labels | choose the threshold on validation data; a percentile flags a fixed share; a lower error is not always a better detector |
| TFLearn to Keras | history, and its Keras mapping | reading old tutorials | its source imports `is_sequence`, which TensorFlow 2.21 no longer defines, so the import is expected to fail: read it, never run it |
| A model as a layer | Keras composition | reusing an encoder or a decoder alone | the decoder's input must match the encoder's output; parts share weights |
| `Sequential` | Keras model | a straight stack with one input and one output | swapped Dense layers build silently; several inputs, outputs or branches need the Functional API |
| TF-IDF into a Dense network | supervised, binary text classifier | routing short texts between two topics | fit the vectorizer on training posts only; the input width must equal the vocabulary size; word order is lost |

**Quick check**
1. What does collaborative filtering need as data, what does an RBM learn from it, and what can it do for a brand-new user?
2. How would you use an autoencoder to flag unusual images, and on which data do you choose the threshold?
3. Why must an autoencoder's bottleneck be smaller than its input, and why is a lower reconstruction error not always better for flagging?
4. What is TFLearn, and what would you use today?
5. What is a Sequential model, and when can you not use one?
6. Why do you fit the TF-IDF vectorizer on the training posts only, and what does a 0.3 threshold do to precision and recall?